<div style="border-left:4px solid #ffffff;padding:10px 18px;margin-bottom:2px;">
<span style="font-size:12pt;font-weight:800;font-family:Georgia,serif;">
Silver Layer · AI4I 2020 · Nettoyage & Standardisation
</span><br>
<span style="font-size:9.5pt;font-family:sans-serif;color:#e8e8e8;">
Nettoyage, validation, renommage, conversion d'unités. Aucun KPI, aucune simulation usine.
Sortie : dataset propre et standardisé prêt pour Gold.
</span>
</div>

<div style="border-left:4px solid #ffffff;padding:10px 18px;margin-bottom:2px;">
<span style="font-size:10pt;font-weight:700;font-family:Georgia,serif;">
00 · Imports & Configuration
</span>
</div>

In [1]:
# ============================================================
# SILVER LAYER — Imports & Configuration
# ============================================================

import pandas as pd
import numpy as np
import json
import logging
from pathlib import Path
from datetime import datetime
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%H:%M:%S'
)
log = logging.getLogger('silver')

ROOT        = Path('..').resolve()
BRONZE_PATH = ROOT / 'data' / 'bronze' / 'ai4i_raw_dirty.csv'
SILVER_PATH = ROOT / 'data' / 'silver'
SILVER_PATH.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

LAYOUT = dict(
    paper_bgcolor='#ffffff',
    plot_bgcolor='#ffffff',
    font=dict(family='Inter, sans-serif', color='#374151', size=11),
    hoverlabel=dict(bgcolor='#1f2937', font_color='#f9fafb', font_size=11),
)

log.info("Configuration Silver chargée. ROOT=%s", ROOT)

11:29:26 | INFO | Configuration Silver chargée. ROOT=C:\Users\j.innouri\OneDrive - pellencst.com\Bureau\MSPR_TPRE831


<div style="border-left:4px solid #ffffff;padding:10px 18px;margin-bottom:2px;">
<span style="font-size:10pt;font-weight:700;font-family:Georgia,serif;">
01 · Chargement depuis Bronze
</span>
</div>

In [2]:
# ============================================================
# CHARGEMENT DEPUIS BRONZE
# On charge le dataset dirty généré en Bronze.
# C'est exactement ce qu'un pipeline Silver reçoit en entrée :
# des données brutes avec des problèmes réels à traiter.
# ============================================================

log.info("Chargement Bronze dirty : %s", BRONZE_PATH)
df = pd.read_csv(BRONZE_PATH)

# On drop les colonnes de métadonnées Bronze — elles ne font pas
# partie de la donnée métier, elles appartiennent à l'ingestion
if 'ingestion_timestamp' in df.columns:
    df = df.drop(columns=['ingestion_timestamp', 'source_file'])

log.info("Chargé : %d lignes · %d colonnes", *df.shape)

print(f"\n{'='*55}")
print(f"  SILVER INPUT — État initial")
print(f"{'='*55}")
print(f"  Lignes    : {df.shape[0]:,}")
print(f"  Colonnes  : {df.shape[1]}")
print(f"  Nulls     : {df.isnull().sum().sum()}")
print(f"  Doublons  : {df.duplicated().sum()}")
print(f"{'='*55}\n")

display(df.head(5))

11:29:26 | INFO | Chargement Bronze dirty : C:\Users\j.innouri\OneDrive - pellencst.com\Bureau\MSPR_TPRE831\data\bronze\ai4i_raw_dirty.csv
11:29:26 | INFO | Chargé : 10020 lignes · 14 colonnes



  SILVER INPUT — État initial
  Lignes    : 10,020
  Colonnes  : 14
  Nulls     : 200
  Doublons  : 20



,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1000,308.6000,1551.0000,42.8000,0,0,0,0,0,0,0
1,2,L47181,L,298.2000,308.7000,1408.0000,46.3000,3,0,0,0,0,0,0
2,3,L47182,L,298.1000,308.5000,1498.0000,49.4000,5,0,0,0,0,0,0
3,4,L47183,L,298.2000,308.6000,1433.0000,39.5000,7,0,0,0,0,0,0
4,5,L47184,L,298.2000,308.7000,1408.0000,40.0000,9,0,0,0,0,0,0


<div style="border-left:4px solid #ffffff;padding:10px 18px;margin-bottom:2px;">
<span style="font-size:10pt;font-weight:700;font-family:Georgia,serif;">
02 · Suppression des doublons
</span>
</div>

In [3]:
# ============================================================
# SUPPRESSION DES DOUBLONS
# Décision Bronze : supprimer les doublons stricts.
# Un doublon strict = toutes les colonnes identiques.
# On garde la première occurrence.
# ============================================================

n_before = len(df)
df = df.drop_duplicates()
n_after  = len(df)
n_removed = n_before - n_after

log.info("Doublons supprimés : %d lignes retirées", n_removed)

print(f"\n  Doublons supprimés : {n_removed} lignes")
print(f"  Lignes restantes   : {n_after:,}")

11:29:26 | INFO | Doublons supprimés : 20 lignes retirées



  Doublons supprimés : 20 lignes
  Lignes restantes   : 10,000


<div style="border-left:4px solid #ffffff;padding:10px 18px;margin-bottom:2px;">
<span style="font-size:10pt;font-weight:700;font-family:Georgia,serif;">
03 · Nettoyage des catégories malformées
</span>
</div>

In [4]:
# ============================================================
# NETTOYAGE CATÉGORIES MALFORMÉES
# Décision Bronze : supprimer les lignes avec Type invalide.
# Valeurs valides : L, M, H uniquement.
# On conserve un log des valeurs supprimées.
# ============================================================

VALID_TYPES = {'L', 'M', 'H'}

invalid_mask  = ~df['Type'].isin(VALID_TYPES)
n_invalid     = invalid_mask.sum()
invalid_vals  = df.loc[invalid_mask, 'Type'].unique().tolist()

df = df[~invalid_mask].copy()

log.info("Catégories invalides supprimées : %d lignes (%s)", n_invalid, invalid_vals)

print(f"\n  Valeurs invalides détectées : {invalid_vals}")
print(f"  Lignes supprimées           : {n_invalid}")
print(f"  Lignes restantes            : {len(df):,}")
print(f"  Distribution Type restante  : {df['Type'].value_counts().to_dict()}")

11:29:26 | INFO | Catégories invalides supprimées : 15 lignes (['MED', '?', 'low', 'X'])



  Valeurs invalides détectées : ['MED', '?', 'low', 'X']
  Lignes supprimées           : 15
  Lignes restantes            : 9,985
  Distribution Type restante  : {'L': 5990, 'M': 2993, 'H': 1002}


<div style="border-left:4px solid #ffffff;padding:10px 18px;margin-bottom:2px;">
<span style="font-size:10pt;font-weight:700;font-family:Georgia,serif;">
04 · Imputation des valeurs manquantes
</span>
</div>

In [5]:
# ============================================================
# IMPUTATION DES VALEURS MANQUANTES
# Décision Bronze : imputation par médiane sur les capteurs.
# La médiane est robuste aux outliers contrairement à la moyenne.
# On impute uniquement les colonnes numériques capteurs.
# Les colonnes binaires (pannes) ne peuvent pas être imputées
# par médiane — on les laisse intactes (elles n'ont pas de nulls).
# ============================================================

COLS_IMPUTE = [
    'Air temperature [K]',
    'Process temperature [K]',
    'Rotational speed [rpm]',
    'Torque [Nm]'
]

imputation_log = {}

for col in COLS_IMPUTE:
    n_null  = df[col].isnull().sum()
    mediane = df[col].median()
    df[col] = df[col].fillna(mediane)
    imputation_log[col] = {'nulls_imputés': int(n_null), 'valeur_médiane': round(mediane, 4)}
    if n_null > 0:
        log.info("Imputation %s : %d nulls → médiane = %.4f", col, n_null, mediane)

print(f"\n  Imputation par médiane :")
for col, info in imputation_log.items():
    print(f"  {col:<35} : {info['nulls_imputés']} nulls → médiane = {info['valeur_médiane']}")

print(f"\n  Nulls restants après imputation : {df.isnull().sum().sum()}")

11:29:26 | INFO | Imputation Air temperature [K] : 50 nulls → médiane = 300.1000
11:29:26 | INFO | Imputation Process temperature [K] : 50 nulls → médiane = 310.1000
11:29:26 | INFO | Imputation Rotational speed [rpm] : 50 nulls → médiane = 1503.0000
11:29:26 | INFO | Imputation Torque [Nm] : 50 nulls → médiane = 40.1000



  Imputation par médiane :
  Air temperature [K]                 : 50 nulls → médiane = 300.1
  Process temperature [K]             : 50 nulls → médiane = 310.1
  Rotational speed [rpm]              : 50 nulls → médiane = 1503.0
  Torque [Nm]                         : 50 nulls → médiane = 40.1

  Nulls restants après imputation : 0


<div style="border-left:4px solid #ffffff;padding:10px 18px;margin-bottom:2px;">
<span style="font-size:10pt;font-weight:700;font-family:Georgia,serif;">
05 · Validation des plages physiques
</span>
</div>

In [6]:
# ============================================================
# VALIDATION DES PLAGES PHYSIQUES
# Décision Bronze : clipping sur la vitesse rotation.
# On applique une politique de clipping plutôt que suppression
# pour conserver le maximum de lignes exploitables.
# Les valeurs aberrantes sont ramenées aux bornes physiques.
# ============================================================

PHYSICAL_RANGES = {
    'Air temperature [K]'      : (290.0, 320.0),
    'Process temperature [K]'  : (300.0, 330.0),
    'Rotational speed [rpm]'   : (0.0,   3000.0),
    'Torque [Nm]'              : (0.0,   80.0),
    'Tool wear [min]'          : (0.0,   300.0),
}

clip_log = {}

print(f"\n  Clipping des plages physiques :")
for col, (lo, hi) in PHYSICAL_RANGES.items():
    n_out = ((df[col] < lo) | (df[col] > hi)).sum()
    df[col] = df[col].clip(lower=lo, upper=hi)
    clip_log[col] = {'valeurs_hors_plage': int(n_out), 'borne_basse': lo, 'borne_haute': hi}
    status = "⚠ corrigé" if n_out > 0 else "✓ OK"
    print(f"  {col:<35} [{lo:>5}, {hi:>6}]  {status}  ({n_out} valeurs)")


  Clipping des plages physiques :
  Air temperature [K]                 [290.0,  320.0]  ✓ OK  (0 valeurs)
  Process temperature [K]             [300.0,  330.0]  ✓ OK  (0 valeurs)
  Rotational speed [rpm]              [  0.0, 3000.0]  ⚠ corrigé  (10 valeurs)
  Torque [Nm]                         [  0.0,   80.0]  ✓ OK  (0 valeurs)
  Tool wear [min]                     [  0.0,  300.0]  ✓ OK  (0 valeurs)


<div style="border-left:4px solid #ffffff;padding:10px 18px;margin-bottom:2px;">
<span style="font-size:10pt;font-weight:700;font-family:Georgia,serif;">
06 · Renommage et conversion d'unités
</span>
</div>

In [7]:
# ============================================================
# RENOMMAGE ET CONVERSION D'UNITÉS
# Convention snake_case, vocabulaire MECHA, unités industrielles.
# Conversion Kelvin → Celsius : T_C = T_K - 273.15
# ============================================================

# Conversion Kelvin → Celsius avant renommage
df['Air temperature [K]']     = (df['Air temperature [K]']     - 273.15).round(2)
df['Process temperature [K]'] = (df['Process temperature [K]'] - 273.15).round(2)

RENAME_MAP = {
    'UDI'                      : 'id',
    'Product ID'               : 'produit_id',
    'Type'                     : 'qualite_produit',
    'Air temperature [K]'      : 'temp_air_c',
    'Process temperature [K]'  : 'temp_process_c',
    'Rotational speed [rpm]'   : 'vitesse_rotation_rpm',
    'Torque [Nm]'              : 'couple_nm',
    'Tool wear [min]'          : 'usure_outil_min',
    'Machine failure'          : 'panne_machine',
    'TWF'                      : 'panne_usure_outil',
    'HDF'                      : 'panne_thermique',
    'PWF'                      : 'panne_puissance',
    'OSF'                      : 'panne_surcharge',
    'RNF'                      : 'panne_aleatoire',
}

df = df.rename(columns=RENAME_MAP)

# Ajout delta thermique — signal précurseur validé en Bronze
df['temp_process_delta'] = (df['temp_process_c'] - df['temp_air_c']).round(2)

log.info("Renommage et conversion terminés. Colonnes : %s", list(df.columns))

print(f"\n  Températures converties : Kelvin → Celsius")
print(f"  temp_air_c      : {df['temp_air_c'].mean():.1f}°C ± {df['temp_air_c'].std():.1f}")
print(f"  temp_process_c  : {df['temp_process_c'].mean():.1f}°C ± {df['temp_process_c'].std():.1f}")
print(f"  delta thermique : {df['temp_process_delta'].mean():.2f} K ± {df['temp_process_delta'].std():.2f}")
print()
print(f"  Colonnes finales Silver : {list(df.columns)}")

11:29:26 | INFO | Renommage et conversion terminés. Colonnes : ['id', 'produit_id', 'qualite_produit', 'temp_air_c', 'temp_process_c', 'vitesse_rotation_rpm', 'couple_nm', 'usure_outil_min', 'panne_machine', 'panne_usure_outil', 'panne_thermique', 'panne_puissance', 'panne_surcharge', 'panne_aleatoire', 'temp_process_delta']



  Températures converties : Kelvin → Celsius
  temp_air_c      : 26.9°C ± 2.0
  temp_process_c  : 36.9°C ± 1.5
  delta thermique : 10.00 K ± 1.01

  Colonnes finales Silver : ['id', 'produit_id', 'qualite_produit', 'temp_air_c', 'temp_process_c', 'vitesse_rotation_rpm', 'couple_nm', 'usure_outil_min', 'panne_machine', 'panne_usure_outil', 'panne_thermique', 'panne_puissance', 'panne_surcharge', 'panne_aleatoire', 'temp_process_delta']


<div style="border-left:4px solid #ffffff;padding:10px 18px;margin-bottom:2px;">
<span style="font-size:10pt;font-weight:700;font-family:Georgia,serif;">
07 · Validation finale & rapport qualité
</span>
</div>

In [8]:
# ============================================================
# VALIDATION FINALE ET RAPPORT QUALITÉ SILVER
# Vérification que toutes les règles de nettoyage ont été
# appliquées correctement avant export.
# ============================================================

quality_report = {
    'timestamp'        : datetime.utcnow().isoformat(),
    'lignes_bronze'    : n_before,
    'doublons_retires' : n_removed,
    'categories_inv'   : n_invalid,
    'lignes_silver'    : len(df),
    'nulls_restants'   : int(df.isnull().sum().sum()),
    'imputation'       : imputation_log,
    'clipping'         : clip_log,
    'colonnes'         : list(df.columns),
}

# Vérifications assert
assert df.isnull().sum().sum() == 0,       "❌ Nulls restants détectés"
assert df.duplicated().sum() == 0,         "❌ Doublons restants détectés"
assert df['qualite_produit'].isin({'L','M','H'}).all(), "❌ Type invalide restant"
assert (df['vitesse_rotation_rpm'] >= 0).all(), "❌ Vitesse négative restante"

print(f"\n{'='*55}")
print(f"  SILVER QUALITY REPORT")
print(f"{'='*55}")
print(f"  Lignes Bronze en entrée  : {quality_report['lignes_bronze']:,}")
print(f"  Doublons retirés         : {quality_report['doublons_retires']}")
print(f"  Catégories invalides     : {quality_report['categories_inv']}")
print(f"  Lignes Silver en sortie  : {quality_report['lignes_silver']:,}")
print(f"  Nulls restants           : {quality_report['nulls_restants']}")
print(f"  Toutes assertions        : ✓ PASSED")
print(f"{'='*55}\n")

display(df.head(10))


  SILVER QUALITY REPORT
  Lignes Bronze en entrée  : 10,020
  Doublons retirés         : 20
  Catégories invalides     : 15
  Lignes Silver en sortie  : 9,985
  Nulls restants           : 0
  Toutes assertions        : ✓ PASSED



,id,produit_id,qualite_produit,temp_air_c,temp_process_c,vitesse_rotation_rpm,couple_nm,usure_outil_min,panne_machine,panne_usure_outil,panne_thermique,panne_puissance,panne_surcharge,panne_aleatoire,temp_process_delta
0,1,M14860,M,24.9500,35.4500,1551.0000,42.8000,0,0,0,0,0,0,0,10.5000
1,2,L47181,L,25.0500,35.5500,1408.0000,46.3000,3,0,0,0,0,0,0,10.5000
2,3,L47182,L,24.9500,35.3500,1498.0000,49.4000,5,0,0,0,0,0,0,10.4000
3,4,L47183,L,25.0500,35.4500,1433.0000,39.5000,7,0,0,0,0,0,0,10.4000
4,5,L47184,L,25.0500,35.5500,1408.0000,40.0000,9,0,0,0,0,0,0,10.5000
5,6,M14865,M,24.9500,35.4500,1425.0000,41.9000,11,0,0,0,0,0,0,10.5000
6,7,L47186,L,24.9500,35.4500,1558.0000,42.4000,14,0,0,0,0,0,0,10.5000
7,8,L47187,L,24.9500,35.4500,1527.0000,40.2000,16,0,0,0,0,0,0,10.5000
8,9,M14868,M,25.1500,35.5500,1667.0000,28.6000,18,0,0,0,0,0,0,10.4000
9,10,M14869,M,26.9500,35.8500,1741.0000,28.0000,21,0,0,0,0,0,0,8.9000


<div style="border-left:4px solid #ffffff;padding:10px 18px;margin-bottom:2px;">
<span style="font-size:10pt;font-weight:700;font-family:Georgia,serif;">
08 · Export Silver
</span>
</div>

In [12]:
# ============================================================
# EXPORT SILVER
# Parquet : format colonnaire efficace, typage strict,
# compression native. Standard industrie pour les couches Silver/Gold.
# On exporte aussi le rapport qualité en JSON.
# ============================================================

silver_file  = SILVER_PATH / 'ai4i_clean.parquet'
report_file  = SILVER_PATH / 'silver_quality_report.json'

df.to_parquet(silver_file, index=False)

def json_serial(obj):
    if isinstance(obj, np.integer): return int(obj)
    if isinstance(obj, np.floating): return float(obj)
    if isinstance(obj, np.ndarray): return obj.tolist()
    raise TypeError(f"Type {type(obj)} non sérialisable")

with open(report_file, 'w') as f:
    json.dump(quality_report, f, indent=2, default=json_serial)

log.info("Silver exporté : %s", silver_file)
log.info("Rapport qualité : %s", report_file)

print(f"\n{'='*55}")
print(f"  SILVER OUTPUTS")
print(f"{'='*55}")
print(f"  ✓ {silver_file}")
print(f"  ✓ {report_file}")
print(f"  Shape final : {df.shape[0]:,} lignes · {df.shape[1]} colonnes")
print(f"{'='*55}")

11:31:49 | INFO | Silver exporté : C:\Users\j.innouri\OneDrive - pellencst.com\Bureau\MSPR_TPRE831\data\silver\ai4i_clean.parquet
11:31:49 | INFO | Rapport qualité : C:\Users\j.innouri\OneDrive - pellencst.com\Bureau\MSPR_TPRE831\data\silver\silver_quality_report.json



  SILVER OUTPUTS
  ✓ C:\Users\j.innouri\OneDrive - pellencst.com\Bureau\MSPR_TPRE831\data\silver\ai4i_clean.parquet
  ✓ C:\Users\j.innouri\OneDrive - pellencst.com\Bureau\MSPR_TPRE831\data\silver\silver_quality_report.json
  Shape final : 9,985 lignes · 15 colonnes
